# Pipeline de datos para clasificacion de letalidad

Este notebook verifica que las tablas procesadas necesarias para el clasificador logistico 2026 existan y tengan la estructura esperada.

In [1]:
from pathlib import Path

import pandas as pd

BASE_DIR = Path('..').resolve()
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'

paths = {
    'event_model_table': PROCESSED_DIR / 'event_model_table.csv',
    'model3_embeddings_dataset': PROCESSED_DIR / 'model3_embeddings_dataset.csv',
}

{name: path.exists() for name, path in paths.items()}

{'event_model_table': True, 'model3_embeddings_dataset': True}

## Carga base

In [2]:
events = pd.read_csv(paths['event_model_table'], low_memory=False)
model_df = pd.read_csv(paths['model3_embeddings_dataset'], low_memory=False)

events['event_date'] = pd.to_datetime(events['event_date'], errors='coerce')
model_df['event_date'] = pd.to_datetime(model_df['event_date'], errors='coerce')
model_df['fatality_positive'] = (pd.to_numeric(model_df['fatalities'], errors='coerce').fillna(0) > 0).astype(int)

events.shape, model_df.shape

((13616, 22), (5120, 65))

## Columnas necesarias

In [3]:
required_columns = [
    'event_id',
    'event_date',
    'fatalities',
    'source',
    'weapon_type',
    'target_type',
    'past_attacks_7d',
    'past_fatalities_7d',
    'military_severity_score',
]

missing = [column for column in required_columns if column not in model_df.columns]
missing

[]

## Snapshot 2026

In [4]:
df_2026 = model_df[model_df['event_date'].dt.year.eq(2026)].copy()

snapshot = {
    'rows_2026': len(df_2026),
    'positive_events': int(df_2026['fatality_positive'].sum()),
    'positive_rate': df_2026['fatality_positive'].mean(),
    'date_min': df_2026['event_date'].min(),
    'date_max': df_2026['event_date'].max(),
}
snapshot

{'rows_2026': 1632,
 'positive_events': 376,
 'positive_rate': np.float64(0.23039215686274508),
 'date_min': Timestamp('2026-02-28 00:00:00'),
 'date_max': Timestamp('2026-05-30 00:00:00')}

## Regeneracion de artefactos

Desde la raiz del proyecto:

```bash
python scripts/prepare_modeling_dataset.py
python scripts/generate_text_embeddings.py
python scripts/train_logreg_2026.py
```